<a href="https://colab.research.google.com/github/nikenyudha/Learn_SQL/blob/main/SQLBASIC_Datatype_and_casting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import library
import sqlite3

# Connect to an SQLite database; use ':memory:' for an in-memory database
conn = sqlite3.connect(':memory:')

In [2]:
c = conn.cursor()
c.execute('''
          CREATE TABLE IF NOT EXISTS orders(
           id       INTEGER,
           amount   TEXT,
           quantity INTEGER)
          ''')

In [3]:
c.execute("INSERT OR IGNORE INTO orders VALUES (1, '150.00', 3)")
c.execute("INSERT OR IGNORE INTO orders VALUES (2, '20.00',  10)")
c.execute("INSERT OR IGNORE INTO orders VALUES (3, '95.50',  1)")
c.execute("INSERT OR IGNORE INTO orders VALUES (4, '200.00', 5)")

In [7]:
c.execute("SELECT id, amount, CAST(amount AS REAL) AS amount_numeric,quantity,CAST(amount AS REAL) * quantity AS total_value FROM orders ORDER BY CAST(amount AS REAL)")
print(c.fetchall())

[(2, '20.00', 20.0, 10, 200.0), (3, '95.50', 95.5, 1, 95.5), (1, '150.00', 150.0, 3, 450.0), (4, '200.00', 200.0, 5, 1000.0)]


In [9]:
import pandas as pd
df2 = pd.read_sql_query("SELECT id, amount, CAST(amount AS REAL) AS amount_numeric,quantity,CAST(amount AS REAL) * quantity AS total_value FROM orders ORDER BY CAST(amount AS REAL)", conn)
print(df2)

   id  amount  amount_numeric  quantity  total_value
0   2   20.00            20.0        10        200.0
1   3   95.50            95.5         1         95.5
2   1  150.00           150.0         3        450.0
3   4  200.00           200.0         5       1000.0


CAST also lets you control precision when storing or displaying values. Casting a REAL to an INTEGER truncates the decimal portion — it does not round.

Date handling is one of the most common places where type awareness pays off. Dates are often imported or received as text strings. Treating them as text means you can concatenate and split them, but you lose the ability to do date arithmetic — calculating the number of days between two events, finding records from the last 30 days, and so on.

In [18]:
c = conn.cursor()
c.execute('''
          CREATE TABLE IF NOT EXISTS signups(
           user_id    INTEGER,
           username   TEXT,
           signed_up  TEXT )
          ''')

In [19]:
c.execute("INSERT OR IGNORE INTO signups VALUES (1, 'alice',   '2025-01-10')")
c.execute("INSERT OR IGNORE INTO signups VALUES (2, 'bob',     '2025-02-28')")
c.execute("INSERT OR IGNORE INTO signups VALUES (3, 'carol',   '2025-03-15')")
c.execute("INSERT OR IGNORE INTO signups VALUES (4, 'dave',    '2024-12-01')")

In [20]:
c.execute("SELECT username, signed_up FROM signups WHERE signed_up >= '2025-01-01' ORDER BY signed_up")
print(c.fetchall())

[('alice', '2025-01-10'), ('bob', '2025-02-28'), ('carol', '2025-03-15')]


In [21]:
import pandas as pd
df2 = pd.read_sql_query("SELECT username, signed_up FROM signups WHERE signed_up >= '2025-01-01' ORDER BY signed_up", conn)
print(df2)

  username   signed_up
0    alice  2025-01-10
1      bob  2025-02-28
2    carol  2025-03-15


ISO 8601 date strings (YYYY-MM-DD) have the useful property that lexicographic ordering matches chronological ordering, so string comparisons work for date ranges. This is not true for MM/DD/YYYY or other regional formats — those must be converted before filtering or sorting.